# 🌌 EinsteinEngine in Action: Schwarzschild Geodesics

In this tutorial, we will use **EinsteinEngine** to derive the equations of motion (geodesics) around a static, spherical Black Hole, described by the Schwarzschild metric.

We will demonstrate the dual-engine architecture of the library:
1. **Theoretical Mode (SymPy):** To extract the formal differential equations and render them in LaTeX.
2. **Numerical Mode (SymEngine C++):** To compile the high-speed algebraic system, ready to be injected into a numerical integrator like `scipy.integrate`.

In [1]:
import sympy as sp
from IPython.display import display, Math

# Import the geometric and physical core of EinsteinEngine
from einsteinengine.symbolic.metric import MetricTensor 
from einsteinengine.symbolic.christoffel import ChristoffelSymbols 
from einsteinengine.symbolic.geodesics import Geodesics 

print("Libraries imported successfully. Ready to curve space-time.")

Libraries imported successfully. Ready to curve space-time.


## 1. Setting the Stage: The Schwarzschild Metric

The first step is to instantiate our space-time. We will use spherical coordinates $(t, r, \theta, \phi)$ and the black hole mass $M$.

The metric is given by:
$$ds^2 = -\left(1 - \frac{2M}{r}\right)dt^2 + \left(1 - \frac{2M}{r}\right)^{-1}dr^2 + r^2 d\theta^2 + r^2 \sin^2(\theta) d\phi^2$$

In [2]:
# Define mathematical symbols
t, r, theta, phi = sp.symbols('t r theta phi', real=True)
M = sp.symbols('M', real=True, positive=True)
syms = [t, r, theta, phi]

# Covariant metric matrix g_uv 
g_schwarzschild = [
    [-(1 - 2*M/r), 0, 0, 0],
    [0, 1/(1 - 2*M/r), 0, 0],
    [0, 0, r**2, 0],
    [0, 0, 0, r**2 * sp.sin(theta)**2]
]

# Initialize the Metric Tensor
metric = MetricTensor(g_schwarzschild, syms, name="Schwarzschild", verbose=True)

[Schwarzschild] Instantiated in 4D spacetime.
[Schwarzschild] Tensor initialized with index configuration: 'll'.
[Schwarzschild] Métrica validada y lista para cálculos.


## 2. Gravitational Forces: Christoffel Symbols

Particles move following geodesics, dictated by the Christoffel Symbols ($\Gamma^\mu_{\alpha\beta}$). The C++ backend of the library will compute the 64 spatial derivatives almost instantly.

In [3]:
# EinsteinEngine automatically computes the inverse matrix and the symbols
christoffel = ChristoffelSymbols.from_metric(metric, verbose=True)

Computing Christoffel Symbols for 'Schwarzschild' in C++...
[Christoffel_Schwarzschild] Instantiated in 4D spacetime.
[Christoffel_Schwarzschild] Connection initialized. Note: This object is NOT a tensor.


## 3. Theoretical Mode: Exact Differential Equations

The default presentation mode generates the Ordinary Differential Equations (ODEs) system. In this mode, the engine understands that the coordinates are functions of proper time $\tau$, i.e., $r(\tau)$.

In [4]:
# Initialize the geodesics generator
geo = Geodesics(christoffel, param_name="tau", verbose=False)

# Render the formal equations on screen
print("Equations of Motion (Theoretical Form):")
geo.display_equations()

Equations of Motion (Theoretical Form):


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 🔬 Theoretical Conclusion
Looking at the equations above, we can notice two major milestones of General Relativity automatically proven by our library:
* In the $\frac{d^2 \phi}{d\tau^2}$ acceleration, the terms depend strictly on the derivatives, which mathematically ensures the **Conservation of Angular Momentum**.
* The radial acceleration component reveals that, unlike Newtonian gravity, space-time curvature interacts with both radial and tangential velocities (this is the relativistic origin of the perihelion precession of Mercury).

## 4. Numerical Mode: Exporting to Simulators (SciPy)

Modern numerical integrators (like Runge-Kutta in Python/C) cannot ingest $\frac{d}{d\tau}$ derivative operators. They require a purely algebraic first-order system.

By setting `substitute_velocities=True`, the engine delegates the computation to **SymEngine (C++)** to compile the tensor contraction using static velocity variables ($v_r, v_\theta$).

In [6]:
# Activate the high-performance numerical backend
num_eqs = geo.get_equations(substitute_velocities=True, simplify=True)

print("Algebraic Accelerations System (Ready for lambdify / SciPy):")
for i, eq in enumerate(num_eqs):
    coord = syms[i].name
    latex_str = f"a_{{{coord}}} = {sp.latex(sp.sympify(eq))}"
    display(Math(latex_str))

Algebraic Accelerations System (Ready for lambdify / SciPy):


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 🚀 Numerical Conclusion
This final block demonstrates the true power of `EinsteinEngine`. The radial equation (the most complex one) has been factored, cleaned, and optimized algebraically. A user now only has to pass this equation through `lambdify` to program a 3D simulation of a spaceship falling into a black hole at thousands of frames per second, completely bypassing the symbolic computation bottleneck.